# AI-детектив · лабораторная №6 · финальный Colab

**Один запуск: Qwen2.5-7B-Instruct через Ollama, A/B/C (36 ответов), обязательные стресс-тесты (84 ответа), отчёт и итоговый ZIP.** Никаких 1.5B/14B, health-check генераций, ablation study или старых результатов.

1. Выбери **Среда выполнения → Сменить среду выполнения → T4 GPU**.
2. Нажми **Выполнить все**. При появлении окна загрузи `lab6_project_clean.zip`.
3. В конце Colab скачает `lab6_submission.zip` с реальными результатами и `report.pdf`. Если прогон неполный, вместо него будет диагностический архив с явной пометкой.

Эталоны основных вопросов и стресс-вмешательств согласованы командой. Перед отправкой заполните фактическое распределение вклада в `CONTRIBUTION.md`, а Streamlit и Git проверьте отдельно.

## 1. Загрузка чистого проекта

Загрузи **только** `lab6_project_clean.zip` (не архив предыдущих результатов). Новая папка создаётся заново; результаты прошлых экспериментов не подмешиваются.

In [ ]:
from google.colab import files
from pathlib import Path
import os, sys, subprocess, json, zipfile, shutil, time, re

uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Загрузите один файл: lab6_project_clean.zip')
archive = Path('/content') / archives[0]
project_root = Path('/content/ai-detective')
with zipfile.ZipFile(archive) as z:
    corrupt = z.testzip()
    if corrupt: raise RuntimeError('Повреждён ZIP: ' + corrupt)
    names = z.namelist()
    if not all('ai-detective/' + f in names for f in ('data/documents.jsonl','data/claims.jsonl','data/questions.jsonl','src/experiment.py','src/stress_tests.py','scripts/finalize_submission.py')):
        raise ValueError('Это не финальный ZIP проекта. Нужен lab6_project_clean.zip')
    root = Path('/content').resolve()
    for item in z.infolist():
        target = (root / item.filename).resolve()
        if target != root and root not in target.parents:
            raise ValueError('Недопустимый путь в ZIP: ' + item.filename)
    if project_root.exists(): shutil.rmtree(project_root)
    z.extractall(root)
PROJECT_ROOT = project_root
(PROJECT_ROOT/'results').mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
sys.path.insert(0,str(PROJECT_ROOT/'src'))
assert not any((PROJECT_ROOT/'results').glob('*.jsonl')), 'В чистом ZIP оказались старые результаты.'
print('Чистый проект загружен:', PROJECT_ROOT)
print('Старых результатов нет.')

## 2. Зависимости и 81 автоматический тест (в чистом Colab-архиве один тест архива результатов пропускается)

В ноутбуке устанавливаются только зависимости, необходимые для основной модели через Ollama, BM25, Streamlit, тестирования и PDF. Тесты не обращаются к модели.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
for script in ('tests/validate_documents.py','tests/validate_claims.py','tests/validate_questions.py'):
    print('Проверка:',script, flush=True)
    subprocess.run([sys.executable,script],check=True,cwd=PROJECT_ROOT)
subprocess.run([sys.executable,'-m','pytest','-q'],check=True,cwd=PROJECT_ROOT)
from common import load_documents,load_claims,load_questions
from stress_tests import build_cases
from collections import Counter
docs,claims,questions = load_documents(),load_claims(),load_questions()
assert (len(docs),len(claims),len(questions))==(10,40,12)
expected_types = Counter(q['question_type'] for q in questions)
assert len(expected_types)==6 and set(expected_types.values())=={2},expected_types
cases=build_cases()
assert len(cases)==84, f'Подготовлено {len(cases)} вместо 84'
print('Документы/утверждения/вопросы:',len(docs),len(claims),len(questions))
print('Стресс-сценарии:',Counter(c['test_type'] for c in cases))

## 3. Единственная модель: Qwen2.5-7B-Instruct

T4 GPU обязательна для разумной работы длинного контекста. Модель загружается через Ollama один раз; API-ключи не используются.

In [ ]:
gpu = subprocess.run(['nvidia-smi'],capture_output=True,text=True)
if gpu.returncode != 0:
    raise RuntimeError('Не обнаружена видеокарта NVIDIA. В Colab выбери T4 GPU и повтори запуск.')
print('GPU доступна:', gpu.stdout.splitlines()[0])
MODEL_NAME = 'qwen2.5:7b-instruct'  # одна модель для всех 120 запросов
os.environ.update(LLM_BACKEND='ollama', OLLAMA_MODEL=MODEL_NAME, OLLAMA_NUM_CTX='32768',
                  OLLAMA_TIMEOUT='900', OLLAMA_MAX_LOADED_MODELS='1', OLLAMA_NUM_PARALLEL='1',
                  OLLAMA_KEEP_ALIVE='30m')
if shutil.which('ollama') is None:
    # Ollama's installer may require zstd to unpack the runtime in Colab.
    subprocess.run(['bash','-lc','apt-get update -qq && apt-get install -y -qq zstd pciutils'],check=True)
    subprocess.run(['bash','-lc','set -euo pipefail; curl -fsSL https://ollama.com/install.sh | sh'],check=True)
import urllib.request

def ollama_ready():
    try:
        with urllib.request.urlopen('http://127.0.0.1:11434/api/tags',timeout=2): return True
    except Exception: return False

if not ollama_ready():
    ollama_log = open('/content/ollama-server.log','a')
    ollama_process=subprocess.Popen(['ollama','serve'],stdout=ollama_log,stderr=ollama_log,
                                     env=os.environ.copy())
    for _ in range(90):
        if ollama_ready(): break
        time.sleep(1)
    else: raise RuntimeError('Ollama не запустилась. Проверь /content/ollama-server.log')
subprocess.run(['ollama','pull',MODEL_NAME],check=True)
from llm import selected_model_name, warmup_model
assert selected_model_name()==MODEL_NAME
print('Основная модель готова:',warmup_model(MODEL_NAME))

## 4. Подготовить доказательства и выполнить базовые 36 ответов

Выполняется только A/B/C на всех 12 вопросах. Результаты записываются в `results/llm_runs.jsonl`. Любые fallback и retry сохраняются отдельно в метаданных.

In [ ]:
subprocess.run([sys.executable,'src/materialize_contexts.py'],check=True,cwd=PROJECT_ROOT)
from evaluation import retrieval_metrics, llm_metrics
rmet = retrieval_metrics()
assert rmet['B_C_same_claim_ids_all_questions'], 'B/C получили разные claim IDs'
from experiment import run_all
base_manifest = run_all(model=MODEL_NAME, temperature=0.0, overwrite=True)
base_rows = [json.loads(x) for x in (PROJECT_ROOT/'results/llm_runs.jsonl').read_text(encoding='utf-8').splitlines() if x.strip()]
print('Базовых результатов:',len(base_rows),'из 36')
print('Статусы валидации:',Counter(x['validation_status'] for x in base_rows))
if len(base_rows)!=36: print('ВНИМАНИЕ: неполный базовый прогон; итоговый архив не будет отмечен как готовый.')

## 5. Полный обязательный стресс-прогон: 84 ответа

**Здесь нет выключающего флага.** Выполняются 36 перестановок (B/C — контроль на том же контексте), 12 удалений необходимых посылок и 36 добавлений противоречий. Каждый сценарий вызывает **эту же Qwen2.5-7B**.

In [ ]:
from stress_tests import run_stress_tests
try:
    stress_manifest = run_stress_tests(run_llm=True, model=MODEL_NAME, temperature=0.0)
except Exception as exc:
    print('Ошибка стресс-прогона:',type(exc).__name__, str(exc))
    print('Частичные результаты будут сохранены в диагностическом архиве.')
stress_path = PROJECT_ROOT/'results/stress_runs.jsonl'
stress_rows = [json.loads(x) for x in stress_path.read_text(encoding='utf-8').splitlines() if x.strip()] if stress_path.exists() else []
print('Фактических стресс-ответов:',len(stress_rows),'из 84')
print('Типы стрессов:',Counter(x['test_type'] for x in stress_rows))
print('Валидация:',Counter(x['validation_status'] for x in stress_rows))
if len(stress_rows)!=84: print('НЕПОЛНЫЙ СТРЕСС: будет только диагностический ZIP, а не отчёт для сдачи.')

## 6. Итоговые метрики и проверка полноты

Сохраняются retrieval, точность статусов, качество доказательств, калибровка и устойчивость. Подготовленные сценарии не выдаются за ответы модели.

In [ ]:
from evaluation import retrieval_metrics, llm_metrics, stress_metrics
import pandas as pd
rmet, lmet, smet = retrieval_metrics(), llm_metrics(), stress_metrics()
assert lmet['total_saved_runs']==36
if smet['actual_stress_runs']!=84: print('ВНИМАНИЕ: данных для окончательного отчёта недостаточно!')
assert rmet['B_C_same_claim_ids_all_questions']
print('Основные метрики (только валидные модельные ответы):')
display(pd.DataFrame([{'режим':m,
                       'model_n':lmet['per_mode'][m]['n_scored_outputs'],
                       'status_accuracy':lmet['per_mode'][m]['status_accuracy'],
                       'evidence_F1':lmet['per_mode'][m]['evidence_f1_macro'],
                       'counter_recall':lmet['per_mode'][m]['counterevidence_recall_macro'],
                       'both_recall':lmet['per_mode'][m]['both_recall'],
                       'insufficient_acc':lmet['per_mode'][m]['insufficient_class_accuracy'],
                       'tokens':lmet['per_mode'][m]['mean_total_tokens'],
                       'seconds':lmet['per_mode'][m]['mean_elapsed_seconds']}
                      for m in 'ABC']))
print('Стресс-метрики:')
display(pd.DataFrame([{'тип':k,**v} for k,v in smet['by_test'].items()]))
print('Всего реально записано: base=',lmet['total_saved_runs'],', stress=',smet['actual_stress_runs'])

## 7. Сформировать финальный отчёт, PDF и ZIP для сдачи

Скрипт создаёт отчёт **только если есть 36 + 84 настоящих записей** и все они структурно валидны (fallback помечается отдельно). Проверяются одна модель, параметры, хеши промптов, полнота и лимит 1500 слов.

Если проверка не пройдена, ноутбук скачает **диагностический**, а не «готовый» ZIP. Отчёт нельзя отправлять преподавателю без проверки командой и заполнения ФИО.

In [ ]:
from google.colab import files
result = subprocess.run([sys.executable,'scripts/finalize_submission.py'],cwd=PROJECT_ROOT,
                        capture_output=True,text=True)
print(result.stdout[-6000:])
if result.returncode == 0:
    final_archive=PROJECT_ROOT.parent/'lab6_submission_final.zip'
    assert final_archive.exists()
    assert (PROJECT_ROOT/'report.pdf').exists()
    print('ФИНАЛЬНЫЙ ЭКСПЕРИМЕНТ ЗАВЕРШЁН. Отчёт и результаты включены в:',final_archive)
    files.download(str(final_archive))
else:
    print('ПРОВЕРКА НЕ ПРОЙДЕНА:',result.stderr[-2500:])
    # Avoid losing the actual model output on a Colab disconnect, but never call it hand-in ready.
    diagnostic = Path('/content/lab6_DIAGNOSTICS_incomplete.zip')
    exclude={'.git','__pycache__','.pytest_cache','.ipynb_checkpoints','.venv'}
    with zipfile.ZipFile(diagnostic,'w',zipfile.ZIP_DEFLATED) as z:
        for p in PROJECT_ROOT.rglob('*'):
            if p.is_file() and not (set(p.relative_to(PROJECT_ROOT).parts) & exclude) and p.suffix!='.pyc':
                z.write(p,arcname=str(Path('ai-detective')/p.relative_to(PROJECT_ROOT)))
    files.download(str(diagnostic))
    print('Скачан архив для диагностики, НЕ для преподавателя.')

## Перед отправкой преподавателю

Комплект: `report.pdf`, результаты в `results/`, код и Streamlit в `lab6_submission.zip`; для Git нужно отдельно опубликовать репозиторий. Эталоны согласованы командой. В `CONTRIBUTION.md` заполните фактическое распределение обязанностей Софьи Атановой, Глеба Гольцмана и Елизаветы Грищук. Затем проверьте Streamlit в браузере и опубликуйте Git-репозиторий. Для демонстрации см. `demo_script.md`. Пришли мне скачанный ZIP, и я проведу последнюю содержательную проверку отчёта и стресс-результатов перед письмом.